In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [2]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC5058/FORS2.2022-01-28T08_17_47.181/NGC-5058_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC5058/FORS2.2022-01-28T08_17_47.181/NGC-5058_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     369   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        50   (2094, 964)   float32   


In [3]:
image_cut = image[65:165, 0:1890]
image_error_cut = image_error[65:165, 0:1890]

In [4]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.00323
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [5]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4, params5):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4) +
        sersic_1d(x_hr, params5)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [6]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC5058/PSF/Real_seeing_NGC5058.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 100, 1)

In [7]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, params5, a, b):
    func = model_convolved(x, params1, params2, params3, params4, params5)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, params5, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    purple_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    choc_area = single_integral(x, params5, a, b)
    sum_area = blue_area + green_area +purple_area + orange_area+ choc_area
    total_area = total_integral(x, params1, params2, params3, params4, params5, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (purple_area*100)/sum_area, (orange_area*100)/sum_area, (choc_area*100)/sum_area ]    

## Halpha

In [8]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))

sky_err = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC5058/Propag_errori/Sky_errors.txt")
real_errors_HA = np.sqrt(radial_profile_error_HA**2+(sky_err[2])**2+(0.04*radial_profile_HA)**2)

fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC5058/FORS2.2022-01-28T08_17_47.181/Fit/Halpha_fit.csv", index_col=0)

In [9]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

3.4571928240659164
3.571313154304204
3.316948678007886
3.347644880791385
1.967894981378873
1.972232293276317
5.882016192641304
5.969197150851867
11.685800143197886
13.630321548551432
              Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       46.241245   4.786991   4.072714   0.922961  1.509840e-01
Green       1.963291  55.693311   4.738293   0.000030  8.138073e-09
Purple     49.232080  37.050633  83.605526  26.549914  4.614848e+00
Orange      2.559009   2.458449   7.510150  67.358183  3.224201e+00
Chocolate   0.004376   0.010617   0.073317   5.168911  9.200997e+01


In [10]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)


df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

8.081375825148404
8.281405462561
15.200896743854708
15.295221918464035
9.812386686519858
9.826845058307368
11.240883681865803
11.37998803177635
47.52205798545086
53.95858028751955
              Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       43.566945   5.344749   4.121742   0.967565  1.860444e-01
Green       2.090040  51.399059   6.363463   0.000040  1.549010e-08
Purple     51.678684  40.542730  81.826911  27.812769  5.682518e+00
Orange      2.659917   2.701457   7.610987  65.395410  4.061323e+00
Chocolate   0.004414   0.012005   0.076896   5.824217  9.007011e+01


In [11]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

10.897908187918546
11.102025729998543
19.849886993349685
19.941417977902464
13.798818832584033
13.814630858158766
18.382251589728465
18.53790604487586
56.37654439055518
63.046776601702504
              Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       40.518422   5.885429   4.145445   1.014939  2.200812e-01
Green       2.938713  47.574393   8.342496   0.000051  2.755597e-08
Purple     53.761636  43.607877  79.766178  29.248641  6.717005e+00
Orange      2.776503   2.918923   7.665530  60.325511  4.926404e+00
Chocolate   0.004726   0.013378   0.080350   9.410858  8.813651e+01


## HBeta

In [12]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
real_errors_HB = np.sqrt(radial_profile_error_HB**2+(sky_err[2])**2+(0.04*radial_profile_HB)**2)
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC5058/FORS2.2022-01-28T08_17_47.181/Fit/Hbeta_fit.csv", index_col = 0)

In [13]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)


df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.3615162471361062
1.2786145613171485
1.1358194769830883
1.1447250158976248
1.689640494530265
1.6895682636828682
3.464549898104992
3.5276282034148445
8.254734066286543
9.372093855837658
              Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       29.420612   0.012802   0.000584   0.000002  8.419771e-08
Green      12.473632  51.708727  13.420655   0.141290  7.556643e-03
Purple     48.338105  39.038463  66.280751  18.545870  5.510641e+00
Orange      9.502705   8.904773  19.324264  75.073605  7.449704e+00
Chocolate   0.264946   0.335236   0.973746   6.239232  8.703210e+01


In [14]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

2.5823016843288475
2.45046314294126
5.375956043772474
5.40397782238199
3.3762388048323206
3.376207031640064
6.6025119192941295
6.703516152304993
14.618614104389923
16.40219946328419
              Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       25.102751   0.019124   0.000631   0.000002  9.729269e-08
Green      13.575419  48.883480  13.888572   0.151682  8.711481e-03
Purple     51.004719  41.292167  65.713533  19.475502  6.227191e+00
Orange     10.036611   9.447558  19.415190  73.599793  8.484985e+00
Chocolate   0.280500   0.357671   0.982073   6.773021  8.527911e+01


In [15]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


4.08172192095202
3.9296471578919485
7.198440282102528
7.226142289911572
5.858469350146737
5.858622160274244
10.656179474228573
10.77277753729767
20.575823184253522
22.631786835377138
              Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       18.734671   0.030608   0.000642   0.000002  1.430970e-07
Green      14.477695  46.433706  13.514609   0.157334  1.267298e-02
Purple     55.591997  43.233015  64.887887  20.789286  7.889770e+00
Orange     10.894284   9.924636  20.526070  69.848043  1.141268e+01
Chocolate   0.301354   0.378035   1.070791   9.205335  8.068488e+01


## NII

In [16]:
mask_NII = (wv > l_NII_2-8)*(wv < l_NII_2+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC5058/FORS2.2022-01-28T08_17_47.181/Fit/NII_fit.csv", index_col = 0)

In [17]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.11931962182303
0.7722977747103423
0.9822042782299271
0.9831487149013891
0.45546069325179933
0.45586304048598153
2.7009579078251664
2.736293643029175
1.4236321198759463
1.5286038842472331
              Peak 1     Peak 2        Peak 3        Peak 4         Peak 5
Blue       55.295808   0.013430  1.933282e-14  1.770488e-27   1.054894e-33
Green       0.152450  35.908338  6.795629e-05  8.333943e-58  6.074627e-121
Purple     36.761782  52.040298  7.660526e+01  1.463868e+01   9.931938e+00
Orange      3.028222   4.931429  1.080269e+01  6.984001e+01   7.650582e+00
Chocolate   4.761738   7.106505  1.259198e+01  1.552131e+01   8.241748e+01


In [18]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

1.988905481972059
1.4384232299918427
1.8724483923643822
1.8723829292416003
2.251588204692358
2.2527125446499703
5.010629266729022
5.067830350796067
6.383904067356966
6.732132671223698
              Peak 1     Peak 2        Peak 3        Peak 4         Peak 5
Blue       49.376008   0.105095  2.474354e-13  4.800125e-27   4.777995e-33
Green       0.435498  32.601392  2.116497e-03  5.308816e-53  3.341850e-107
Purple     41.406376  54.637583  7.616103e+01  1.579265e+01   1.108729e+01
Orange      3.415763   5.188758  1.104397e+01  6.728939e+01   8.771247e+00
Chocolate   5.366354   7.467173  1.279287e+01  1.691796e+01   8.014147e+01


In [19]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

3.002992698267635
2.3734012585312882
3.0513996617060357
3.0431877269670493
3.1225034688508453
3.1234289096369636
7.756586612266746
7.824357784026728
8.218322991900099
8.579574468377295
              Peak 1     Peak 2        Peak 3        Peak 4         Peak 5
Blue       37.361245   0.516961  1.651467e-12  1.522058e-26   1.588332e-32
Green       3.213450  25.546076  1.286386e-02  2.342980e-48  2.261613e-100
Purple     48.952349  59.885385  7.572729e+01  1.757066e+01   1.207293e+01
Orange      4.095768   5.803336  1.128185e+01  6.140811e+01   9.850554e+00
Chocolate   6.377187   8.248242  1.297800e+01  2.102124e+01   7.807651e+01


## SII

In [20]:
mask_SII = (wv > l_SII_1-8)*(wv < l_SII_1+8)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC5058/FORS2.2022-01-28T08_17_47.181/Fit/SII_fit.csv", index_col = 0)

In [21]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)


df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.6329432401151414
0.7064467599208379
1.261478736595625
1.2638119789168825
1.1540058982987547
1.1548966447787028
2.8287015323473796
2.963782860910836
2.670408941045227
2.7188787716901164
              Peak 1     Peak 2     Peak 3        Peak 4        Peak 5
Blue       10.754647   0.004133   0.000473  2.134482e-05  9.464403e-06
Green       0.305254  38.069963   0.110955  1.157702e-13  8.549193e-23
Purple     71.188695  48.939735  76.198089  1.634508e+01  1.190102e+01
Orange      3.837223   3.181401   7.095348  6.221810e+01  7.097357e+00
Chocolate  13.914182   9.804768  16.595135  2.143680e+01  8.100162e+01


In [22]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

1.2417634983989987
1.3576131603151578
3.0131925412677845
3.017471808386384
2.871779952084108
2.873410525965323
5.313991918270121
5.530521499944779
5.101099691954482
5.178300407384858
              Peak 1     Peak 2     Peak 3        Peak 4        Peak 5
Blue        8.748414   0.004204   0.000451  2.286103e-05  9.947043e-06
Green       0.498779  33.628956   0.146858  4.283644e-13  4.310731e-22
Purple     72.632774  52.377392  75.119814  1.741621e+01  1.246952e+01
Orange      3.920479   3.467863   7.553103  5.959496e+01  7.544160e+00
Chocolate  14.199554  10.521585  17.179775  2.298880e+01  7.998631e+01


In [23]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

2.0814969406303008
2.213422521654137
4.009845544819398
4.0128868715755
3.978754232492821
3.980126457622238
8.407277963353177
8.659078678986013
8.259837779575381
8.346879815735775
              Peak 1     Peak 2     Peak 3        Peak 4        Peak 5
Blue        6.037517   0.005738   0.000474  2.447814e-05  1.130222e-05
Green       0.825427  29.999054   0.297285  1.779398e-12  2.205673e-20
Purple     74.563599  55.222334  74.540278  1.894218e+01  1.375456e+01
Orange      4.001438   3.673167   7.732472  5.376698e+01  9.197711e+00
Chocolate  14.572019  11.099706  17.429490  2.729082e+01  7.704772e+01


## OIII

In [24]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC5058/FORS2.2022-01-28T08_17_47.181/Fit/OIII_fit.csv", index_col = 0)

In [25]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)


df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

3.9735829972965613
3.9730302487432874
3.9469221949849786
4.208559685041918
3.963870419222013
3.96529063648929
4.984059297713034
4.872044732098561
22.280502711094403
25.435912421370023
              Peak 1     Peak 2     Peak 3        Peak 4        Peak 5
Blue       69.515904  10.304462  11.829873  3.365274e+00  4.881792e-01
Green      19.720855  73.995112  28.621337  2.116270e+00  2.002804e-01
Purple      0.000117   7.454602  35.818019  2.428452e-08  2.491418e-84
Orange     10.730258   8.204234  23.543481  9.031008e+01  7.270804e+00
Chocolate   0.032866   0.041590   0.187290  4.208373e+00  9.204074e+01


In [26]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

7.638357300002749
7.63610036751521
17.72672817298928
18.58878510523742
8.003352688323275
8.006622005750934
9.568057584051855
9.389860186650388
38.121449671277304
43.15549712421557
              Peak 1     Peak 2     Peak 3        Peak 4        Peak 5
Blue       67.795233  11.658671  11.765527  3.511206e+00  5.712570e-01
Green      20.990248  70.575907  29.345032  2.217184e+00  2.349828e-01
Purple      0.000712   8.553718  35.321114  6.600971e-07  4.287591e-69
Orange     11.179411   9.164718  23.380811  8.965950e+01  8.541128e+00
Chocolate   0.034397   0.046985   0.187517  4.612105e+00  9.065263e+01


In [27]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

12.185181439374004
12.176331601097452
22.861094340718285
23.75149012392058
13.939427196480256
13.949089979547232
17.364202869767993
17.151829472139738
50.2551364469962
56.06722532080457
              Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       65.561031  12.893202  11.524022   3.892683  7.761624e-01
Green      22.369467  67.547611  29.026569   2.499281  3.278751e-01
Purple      0.003928   9.521222  34.846302   0.000120  2.710965e-45
Orange     12.029119   9.986137  24.395213  87.291939  1.197953e+01
Chocolate   0.036455   0.051828   0.207895   6.315977  8.691643e+01
